In [1]:
# ⚙️ 1. Install Required Libraries
# %pip install pymupdf tqdm beautifulsoup4 pandas pyarrow


In [ ]:
# 🧩 2. Fast Multi-Threaded PDF → Text Conversion
import os
import fitz  # PyMuPDF
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import re

start = int(input("Enter Start Year: "))
end = int(input("Enter Start Year: "))



In [ ]:
BASE_DIR = f"D:\LPA_MTech_Project\My_Datasets\SC_{start}-{end}"
OUT_DIR = f"D:\LPA_MTech_Project\Extracted_Texts\Texts_{start}-{end}"
os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
# === Text Cleaning Function ===
def clean_case_text(text):
    # 1️⃣ Remove repeated alphabet ladders (A–H blocks)
    text = re.sub(r'(\b[A-H]\b[\s\n]*){3,}', ' ', text)

    # 2️⃣ Remove numeric-only lines (page numbers)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)

    # 3️⃣ Remove known SC headers/footers (but keep legal terms inside text)
    remove_patterns = [
        r"^\s*SUPREME COURT REPORTS\s*$",
        r"^\s*\[\d{4}\]\s*\d+\s*S\.C\.R\.\s*$",   # e.g. [2020] 1 S.C.R.
        r"^\s*REPORTABLE\s*$",
        r"^\s*NON-REPORTABLE\s*$",
        r"^\s*JUDGMENT\s*$",
        r"^\s*ORDER\s*$",
        r"^\s*CORAM\s*:.*$",                     # judge listing line
        r"^\s*BENCH\s*:.*$",                     # bench composition
    ]
    for p in remove_patterns:
        text = re.sub(p, " ", text, flags=re.IGNORECASE | re.MULTILINE)

    # 4️⃣ Remove dotted or dashed separators (---, …, etc.)
    text = re.sub(r"[-]{2,}|[.]{3,}|[_]{2,}", " ", text)

    # 5️⃣ Remove stray page titles like "SUPREME COURT REPORTS [2020] 1 S.C.R."
    text = re.sub(r"SUPREME COURT REPORTS\s*\[\d{4}\]\s*\d+\s*S\.C\.R\.", " ", text, flags=re.IGNORECASE)

    # 6️⃣ Normalize whitespace
    text = re.sub(r'\s{2,}', ' ', text)
    text = re.sub(r'\n{2,}', '\n', text)

    return text.strip()

In [4]:
# === PDF → Text Extraction Function ===
def extract_text_fitz(pdf_path, txt_path):
    try:
        with fitz.open(pdf_path) as doc:
            text = "".join([page.get_text("text") for page in doc])
        text = clean_case_text(text)
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(text)
    except Exception as e:
        print(f"⚠️ Error extracting {pdf_path}: {e}")

In [ ]:
# === Collect All PDFs Across Years ===
pdf_files = []
for year in range(start, end+1):
    year_path = os.path.join(BASE_DIR, str(year), "english")
    if os.path.exists(year_path):
        pdfs = [os.path.join(year_path, f) for f in os.listdir(year_path) if f.endswith(".pdf")]
        pdf_files.extend(pdfs)
        print(f"✅ {len(pdfs)} PDFs found in {year_path}")

print(f"\n📄 Total PDFs found: {len(pdf_files)}")

✅ 401 PDFs found in D:\LPA_MTech_Project\My_Datasets\SC_1990-2009\1990\english
✅ 350 PDFs found in D:\LPA_MTech_Project\My_Datasets\SC_1990-2009\1991\english
✅ 362 PDFs found in D:\LPA_MTech_Project\My_Datasets\SC_1990-2009\1992\english
✅ 397 PDFs found in D:\LPA_MTech_Project\My_Datasets\SC_1990-2009\1993\english
✅ 624 PDFs found in D:\LPA_MTech_Project\My_Datasets\SC_1990-2009\1994\english
✅ 917 PDFs found in D:\LPA_MTech_Project\My_Datasets\SC_1990-2009\1995\english
✅ 1536 PDFs found in D:\LPA_MTech_Project\My_Datasets\SC_1990-2009\1996\english
✅ 846 PDFs found in D:\LPA_MTech_Project\My_Datasets\SC_1990-2009\1997\english
✅ 494 PDFs found in D:\LPA_MTech_Project\My_Datasets\SC_1990-2009\1998\english
✅ 570 PDFs found in D:\LPA_MTech_Project\My_Datasets\SC_1990-2009\1999\english
✅ 610 PDFs found in D:\LPA_MTech_Project\My_Datasets\SC_1990-2009\2000\english
✅ 612 PDFs found in D:\LPA_MTech_Project\My_Datasets\SC_1990-2009\2001\english
✅ 578 PDFs found in D:\LPA_MTech_Project\My_Dataset

In [6]:
# === Parallel Extraction ===
with ThreadPoolExecutor(max_workers=6) as executor:
    list(tqdm(
        executor.map(
            lambda p: extract_text_fitz(
                p,
                os.path.join(OUT_DIR, os.path.basename(p).replace('.pdf', '.txt'))
            ),
            pdf_files
        ),
        total=len(pdf_files),
        desc="Extracting & Cleaning PDFs"
    ))

print(f"\n✅ All PDFs converted and cleaned → saved in: {OUT_DIR}")

Extracting & Cleaning PDFs: 100%|██████████| 14218/14218 [38:50<00:00,  6.10it/s]  


✅ All PDFs converted and cleaned → saved in: D:\LPA_MTech_Project\Extracted_Texts\Texts_1990-2009
